# Face Mask Detection — MobileNetV2 training

Run this notebook in Google Colab with **Runtime → Change runtime type → T4 GPU** (or another available GPU). It downloads the public Kaggle dataset `omkargurav/face-mask-dataset` with KaggleHub unless `DATASET_ROOT` is set to a local/Drive copy. The dataset is not included in this repository. Training, evaluation, plots, and the `.keras` model are produced only by running the corresponding cells; no metrics or model are pre-generated.

## 1. Install and import libraries

In [ ]:
%pip -q install kagglehub scikit-learn

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.mobilenet_v2 import (MobileNetV2,
                                                        preprocess_input)
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print('TensorFlow:', tf.__version__)
gpu_devices = tf.config.list_physical_devices('GPU')
print('GPU devices:', gpu_devices if gpu_devices else 'None — training will use CPU')

## 2. Dataset paths and class mapping

The expected layout is `dataset/with_mask/` and `dataset/without_mask/`. By default the notebook downloads the public Kaggle dataset. For a Drive or manually downloaded copy, set `DATASET_ROOT` below to the directory containing those two class folders (or their parent directory). The mapping is deliberately fixed: **0 = with_mask, 1 = without_mask**.

In [ ]:
DATASET_ROOT = None  # Example: '/content/drive/MyDrive/dataset'

if DATASET_ROOT is None:
    import kagglehub
    DATASET_ROOT = kagglehub.dataset_download('omkargurav/face-mask-dataset')

DATASET_ROOT = Path(DATASET_ROOT).expanduser().resolve()
CLASS_TO_INDEX = {'with_mask': 0, 'without_mask': 1}
INDEX_TO_CLASS = {index: name for name, index in CLASS_TO_INDEX.items()}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Dataset directory does not exist: {DATASET_ROOT}')

candidate_dirs = [DATASET_ROOT, *(p for p in DATASET_ROOT.rglob('*') if p.is_dir())]
matching_parents = [p for p in candidate_dirs
                    if all((p / class_name).is_dir() for class_name in CLASS_TO_INDEX)]
if not matching_parents:
    raise FileNotFoundError(
        f'Could not find both class folders under {DATASET_ROOT}. '
        'Expected with_mask/ and without_mask/.')
CLASS_DIRS = min(matching_parents, key=lambda p: len(p.parts))
print('Using dataset:', CLASS_DIRS)

## 3. Explore the dataset

Count supported image files in both class folders and display a few real examples. The notebook stops if either class is empty.

In [ ]:
records = []
for class_name, label in CLASS_TO_INDEX.items():
    image_paths = sorted(p for p in (CLASS_DIRS / class_name).rglob('*')
                         if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not image_paths:
        raise ValueError(f'No supported images found in {CLASS_DIRS / class_name}')
    records.extend({'path': str(path), 'class_name': class_name, 'label': label}
                   for path in image_paths)

data = pd.DataFrame(records)
class_counts = data['class_name'].value_counts().reindex(CLASS_TO_INDEX.keys(), fill_value=0)
print('Total images:', len(data))
print('with_mask:', int(class_counts['with_mask']))
print('without_mask:', int(class_counts['without_mask']))
display(class_counts.rename('image_count').to_frame())

sample_data = data.sample(n=min(8, len(data)), random_state=SEED)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, row in zip(axes.flat, sample_data.itertuples()):
    image = tf.keras.utils.load_img(row.path, target_size=(224, 224), color_mode='rgb')
    ax.imshow(image)
    ax.set_title(f'{row.class_name} (label {row.label})')
    ax.axis('off')
for ax in axes.flat[len(sample_data):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Reproducible train/validation/test split

Split image paths before training augmentation, using stratification and a fixed seed. Each file belongs to exactly one partition: 70% training, 15% validation, and 15% held-out testing. Test images are not augmented.

In [ ]:
train_validation, test_data = train_test_split(
    data, test_size=0.15, random_state=SEED, stratify=data['label'])
train_data, validation_data = train_test_split(
    train_validation, test_size=(0.15 / 0.85), random_state=SEED,
    stratify=train_validation['label'])

assert set(train_data['path']).isdisjoint(validation_data['path'])
assert set(train_data['path']).isdisjoint(test_data['path'])
assert set(validation_data['path']).isdisjoint(test_data['path'])
print(f'Train: {len(train_data)} ({len(train_data) / len(data):.1%})')
print(f'Validation: {len(validation_data)} ({len(validation_data) / len(data):.1%})')
print(f'Test: {len(test_data)} ({len(test_data) / len(data):.1%})')
split_counts = pd.DataFrame({
    'train': train_data['label'].value_counts().reindex([0, 1], fill_value=0).to_numpy(),
    'validation': validation_data['label'].value_counts().reindex([0, 1], fill_value=0).to_numpy(),
    'test': test_data['label'].value_counts().reindex([0, 1], fill_value=0).to_numpy(),
}, index=['with_mask (0)', 'without_mask (1)'])
display(split_counts)

## 5. Image preprocessing and training-only augmentation

Decode images as RGB, resize to **224 × 224 × 3**, then use `mobilenet_v2.preprocess_input` to scale pixels to the range **[-1, 1]**. Random horizontal flip, small rotation, zoom, and translation are model layers and are active only during training; validation and test inputs remain deterministic.

In [ ]:
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_and_preprocess(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = preprocess_input(image)
    return image, tf.cast(label, tf.float32)

def make_dataset(frame, training=False):
    ds = tf.data.Dataset.from_tensor_slices((
        frame['path'].to_numpy(), frame['label'].to_numpy(dtype=np.float32)))
    if training:
        ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_data, training=True)
validation_ds = make_dataset(validation_data)
test_ds = make_dataset(test_data)

## 6. Build and train MobileNetV2

Use ImageNet-pretrained MobileNetV2 without its top, freeze it for the initial training phase, then optionally fine-tune its last layers at a low learning rate. The sigmoid output estimates the probability of class 1 (`without_mask`); a threshold of 0.5 maps predictions to the fixed class IDs.

In [ ]:
PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'ml'
MODEL_DIR = OUTPUT_DIR / 'model'
RESULTS_DIR = OUTPUT_DIR / 'results'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
HEAD_CHECKPOINT_PATH = RESULTS_DIR / 'head_best.keras'
FINE_TUNE_CHECKPOINT_PATH = RESULTS_DIR / 'fine_tune_best.keras'

augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.04),
    layers.RandomZoom(0.08),
    layers.RandomTranslation(0.05, 0.05),
], name='training_augmentation')

base_model = MobileNetV2(weights='imagenet', include_top=False,
                         input_shape=(224, 224, 3))
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3), name='rgb_preprocessed_image')
x = augmentation(inputs)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation='sigmoid', name='without_mask_probability')(x)
model = tf.keras.Model(inputs, outputs, name='mobilenetv2_mask_detector')

def compile_model(learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.BinaryAccuracy(name='accuracy')])

def make_callbacks(checkpoint_path):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                         restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint(str(checkpoint_path), monitor='val_loss',
                                           save_best_only=True, verbose=1),
    ]

compile_model(1e-3)
model.summary()
history = model.fit(train_ds, validation_data=validation_ds, epochs=15,
                    callbacks=make_callbacks(HEAD_CHECKPOINT_PATH))

In [ ]:
# Fine-tune the last 30 backbone layers, excluding BatchNormalization layers.
FINE_TUNE = True
history_data = {key: list(values) for key, values in history.history.items()}

if FINE_TUNE:
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(1e-5)
    fine_tune_history = model.fit(
        train_ds, validation_data=validation_ds, epochs=5,
        callbacks=make_callbacks(FINE_TUNE_CHECKPOINT_PATH))
    for key, values in fine_tune_history.history.items():
        history_data.setdefault(key, []).extend(values)

# Compare each phase's best checkpoint so fine-tuning cannot displace a better head-only model.
checkpoint_candidates = [(min(history.history['val_loss']), HEAD_CHECKPOINT_PATH)]
if FINE_TUNE and FINE_TUNE_CHECKPOINT_PATH.exists():
    checkpoint_candidates.append((min(fine_tune_history.history['val_loss']),
                                  FINE_TUNE_CHECKPOINT_PATH))
best_validation_loss, best_checkpoint_path = min(checkpoint_candidates, key=lambda item: item[0])
model = tf.keras.models.load_model(best_checkpoint_path)
print(f'Best validation loss: {best_validation_loss:.4f}')
print('Best validation-loss checkpoint loaded:', best_checkpoint_path)

## 7. Plot actual training and validation history

These figures are generated from the `model.fit` history in this run and saved under `ml/results/`.

In [ ]:
epochs_ran = range(1, len(history_data['accuracy']) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_data['accuracy'], label='Training Accuracy')
plt.plot(epochs_ran, history_data['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'accuracy.png', dpi=150)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(epochs_ran, history_data['loss'], label='Training Loss')
plt.plot(epochs_ran, history_data['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'loss.png', dpi=150)
plt.show()

## 8. Held-out test evaluation

Report accuracy, precision, recall, and F1-score from the untouched test partition. The confusion matrix uses the same explicit order and labels as the fixed class mapping.

In [ ]:
y_true = test_data['label'].to_numpy(dtype=np.int32)
y_probability = model.predict(test_ds, verbose=1).ravel()
y_pred = (y_probability >= 0.5).astype(np.int32)

test_metrics = {
    'Test Accuracy': accuracy_score(y_true, y_pred),
    'Precision': precision_score(y_true, y_pred, zero_division=0),
    'Recall': recall_score(y_true, y_pred, zero_division=0),
    'F1-score': f1_score(y_true, y_pred, zero_division=0),
}
for metric_name, value in test_metrics.items():
    print(f'{metric_name}: {value:.4f}')

matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
display = ConfusionMatrixDisplay(
    confusion_matrix=matrix,
    display_labels=[INDEX_TO_CLASS[0], INDEX_TO_CLASS[1]])
display.plot(cmap='Blues', values_format='d')
plt.title('Held-out Test Confusion Matrix')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

## 9. Save the trained model and inspect sample predictions

The saved model includes its architecture and trained weights. Its input contract is an RGB float image resized to 224 × 224 and preprocessed with `mobilenet_v2.preprocess_input` (values in [-1, 1]); output is the sigmoid probability for `without_mask` (class 1).

In [ ]:
FINAL_MODEL_PATH = MODEL_DIR / 'mask_detector.keras'
model.save(FINAL_MODEL_PATH)
for checkpoint_path in (HEAD_CHECKPOINT_PATH, FINE_TUNE_CHECKPOINT_PATH):
    if checkpoint_path.exists():
        checkpoint_path.unlink()
print('Trained model saved:', FINAL_MODEL_PATH.resolve())

sample_rows = test_data.sample(n=min(6, len(test_data)), random_state=SEED)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, row in zip(axes.flat, sample_rows.itertuples()):
    raw_image = tf.keras.utils.load_img(row.path, target_size=IMAGE_SIZE, color_mode='rgb')
    image_array = tf.keras.utils.img_to_array(raw_image)
    input_tensor = preprocess_input(image_array.copy())[None, ...]
    without_mask_probability = float(model.predict(input_tensor, verbose=0)[0, 0])
    predicted_label = int(without_mask_probability >= 0.5)
    confidence = (without_mask_probability if predicted_label == 1
                  else 1.0 - without_mask_probability)
    ax.imshow(raw_image)
    ax.set_title(
        f'Actual: {INDEX_TO_CLASS[int(row.label)]}\n'
        f'Predicted: {INDEX_TO_CLASS[predicted_label]} ({confidence:.1%})')
    ax.axis('off')
for ax in axes.flat[len(sample_rows):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

print('Artifacts:', FINAL_MODEL_PATH, RESULTS_DIR / 'accuracy.png',
      RESULTS_DIR / 'loss.png', RESULTS_DIR / 'confusion_matrix.png')

## 10. Export results from Colab

When run in Colab, download a ZIP containing the trained model and the three actual result plots, then place the files in the same relative locations in the repository. The dataset itself is intentionally excluded.

In [ ]:
import zipfile

artifact_zip = OUTPUT_DIR / 'mask_detection_artifacts.zip'
with zipfile.ZipFile(artifact_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(FINAL_MODEL_PATH, 'ml/model/mask_detector.keras')
    for filename in ('accuracy.png', 'loss.png', 'confusion_matrix.png'):
        archive.write(RESULTS_DIR / filename, f'ml/results/{filename}')
print('Artifact archive created:', artifact_zip.resolve())

if 'google.colab' in str(get_ipython().__class__).lower():
    from google.colab import files
    files.download(str(artifact_zip))